# Breast Cancer Diagnostic Data Analysis

**Statistical analysis and classification on a real clinical dataset**

This notebook analyzes the **Breast Cancer Wisconsin (Diagnostic)** dataset:
569 real patient samples, each describing 30 numeric features computed from
a digitized image of a fine needle aspirate (FNA) of a breast mass (cell
nucleus radius, texture, perimeter, smoothness, concavity, etc.), with a
diagnosis label of **malignant** or **benign**.

This is real, publicly available clinical data (distributed with
scikit-learn; originally from Dr. William H. Wolberg, University of
Wisconsin Hospitals), not simulated data.

**Structure:**
1. Exploratory data analysis
2. Hypothesis testing — do malignant and benign tumors differ statistically?
3. Logistic regression classifier
4. Model evaluation & feature importance


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, precision_recall_fscore_support,
)

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
%matplotlib inline

df = pd.read_csv("../data/breast_cancer_diagnostic.csv")
df.head()

## 1. Exploratory data analysis

The dataset has 30 features: the mean, standard error, and "worst"
(largest) value of 10 cell-nucleus characteristics measured per image.
We'll start with the 10 mean features, class balance, and correlation
structure.

In [ ]:
KEY_FEATURES = [
    "mean_radius", "mean_texture", "mean_perimeter", "mean_area",
    "mean_smoothness", "mean_compactness", "mean_concavity",
    "mean_concave_points", "mean_symmetry", "mean_fractal_dimension",
]

print(df["diagnosis"].value_counts())
df[KEY_FEATURES].describe().round(3)

In [ ]:
plt.figure(figsize=(5.5, 5))
sns.countplot(data=df, x="diagnosis", hue="diagnosis",
              palette={"benign": "#4C72B0", "malignant": "#C44E52"}, legend=False)
plt.title("Class balance")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, feat in zip(axes.flat, ["mean_radius", "mean_texture", "mean_concavity", "mean_area"]):
    sns.kdeplot(data=df, x=feat, hue="diagnosis", fill=True, alpha=0.4, ax=ax,
                palette={"benign": "#4C72B0", "malignant": "#C44E52"})
    ax.set_title(feat.replace("_", " "))
fig.suptitle("Distribution of key features by diagnosis")
plt.tight_layout()
plt.show()

In [ ]:
corr = df[KEY_FEATURES].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True,
            xticklabels=True, yticklabels=True, cbar_kws={"shrink": 0.8})
plt.title("Correlation among mean cell-nucleus features")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

**Observation:** malignant tumors visually skew toward larger radius,
higher texture variance, and higher concavity. Size-related features
(radius, perimeter, area) are highly correlated with each other, which
makes sense geometrically — worth keeping in mind for the model, since
correlated features can make individual coefficients harder to interpret
in isolation.

## 2. Hypothesis testing

For each mean feature, we run a **Welch's t-test** (does not assume equal
variances) comparing malignant vs. benign tumors: is the difference in
means larger than we'd expect by chance?

In [ ]:
rows = []
for feat in KEY_FEATURES:
    mal = df.loc[df["diagnosis"] == "malignant", feat]
    ben = df.loc[df["diagnosis"] == "benign", feat]
    t_stat, p_val = stats.ttest_ind(mal, ben, equal_var=False)
    rows.append({"feature": feat, "malignant_mean": mal.mean(),
                 "benign_mean": ben.mean(), "t_stat": t_stat, "p_value": p_val})

test_df = pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)
test_df

In [ ]:
n_sig = (test_df["p_value"] < 0.05).sum()
print(f"{n_sig} of {len(test_df)} features differ significantly (p < .05) "
      "between malignant and benign tumors.")

plt.figure(figsize=(9, 6))
sns.boxplot(data=df, x="diagnosis", y="mean_concave_points", hue="diagnosis",
            palette={"benign": "#4C72B0", "malignant": "#C44E52"}, legend=False)
plt.title("Mean concave points by diagnosis\n(largest group difference, t-test)")
plt.tight_layout()
plt.show()

**Observation:** 9 of 10 mean features differ significantly between
malignant and benign tumors — only `mean_fractal_dimension` shows no
significant difference. `mean_concave_points` shows the largest separation,
consistent with the clinical intuition that irregular, concave cell
boundaries are a hallmark of malignancy.

## 3. Logistic regression classifier

Now we move from *describing* differences to *predicting* diagnosis. We
use all 30 features, an 75/25 stratified train/test split, and standardize
features before fitting logistic regression.

In [ ]:
feature_cols = [c for c in df.columns if c not in ("target", "diagnosis")]
X = df[feature_cols]
y = (df["diagnosis"] == "malignant").astype(int)  # 1 = malignant

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(max_iter=5000, random_state=42)
clf.fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)
y_prob = clf.predict_proba(X_test_s)[:, 1]

print(classification_report(y_test, y_pred, target_names=["benign", "malignant"]))

## 4. Model evaluation & feature importance

In [ ]:
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["benign", "malignant"], yticklabels=["benign", "malignant"])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title(f"Confusion matrix (accuracy = {acc:.3f})")

fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color="crimson", linewidth=2, label=f"ROC (AUC = {auc:.3f})")
axes[1].plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1)
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curve")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
coef = pd.Series(clf.coef_[0], index=feature_cols, name="coefficient")
top_coef = coef.reindex(coef.abs().sort_values(ascending=False).index).head(12)

plt.figure(figsize=(9, 7))
colors = ["#C44E52" if v > 0 else "#4C72B0" for v in top_coef.values]
plt.barh(top_coef.index[::-1], top_coef.values[::-1], color=colors[::-1])
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Top 12 features by logistic regression coefficient\n(red = pushes toward malignant)")
plt.xlabel("Standardized coefficient")
plt.tight_layout()
plt.show()

## 5. Takeaways

- 9 of 10 mean cell-nucleus features differ significantly between malignant
  and benign tumors — malignancy is strongly associated with larger,
  more irregular (concave, high-texture-variance) cell nuclei.
- A logistic regression classifier using all 30 features achieves
  **~96–97% accuracy** and **AUC ≈ 0.996** on held-out test data — strong
  performance, consistent with this being one of the more separable
  classic benchmark datasets in the ML literature.
- The features with the largest standardized coefficients (`worst_texture`,
  `radius_error`, `worst_symmetry`, `mean_concave_points`) point to the same
  story as the hypothesis tests: irregularity and size variability are the
  dominant signals.

### Limitations

- This is a well-studied benchmark dataset; real-world diagnostic accuracy
  depends heavily on imaging quality, population, and clinical context —
  this notebook is a statistics/ML exercise, not a diagnostic tool.
- Many features are highly correlated (radius/perimeter/area), so
  individual coefficients should be read as part of a correlated group,
  not fully independent effects.
